# Primeiro modelo

## Célula 1 — Instalar dependências

In [ ]:
!pip install langchain langchain-community langchain-huggingface sentence-transformers pypdf

## Célula 2 — Importar bibliotecas



In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

## Célula 3 — Carregar documento

In [ ]:
file_path = "/content/TCC  versão 4.6.pdf"

loader = PyPDFLoader(file_path)
docs = loader.load()

print("Número de páginas carregadas:", len(docs))

## Célula 4 — Dividir texto em chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=200,
    add_start_index=True
)

all_splits = text_splitter.split_documents(docs)

print("Número de chunks:", len(all_splits))

## Célula 5 — Criar embeddings com SERAFIM

In [ ]:
embeddings = HuggingFaceEmbeddings(
    #model_name="alfaneo/bertimbau-base-portuguese-sts"
     model_name="PORTULAN/serafim-900m-portuguese-pt-sentence-encoder"
)

## Célula 6 — Criar vector store

In [ ]:
vector_store = InMemoryVectorStore(embeddings)

ids = vector_store.add_documents(documents=all_splits)

print("Documentos indexados:", len(ids))

## Célula 7 — Busca semântica

In [ ]:
query = "Qual é o assunto principal do documento?"

results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\nResultado {i+1}")
    print(doc.page_content[:500])

## Célula 8 — Criar retriever

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

## Célula 9 — Fazer perguntas

In [ ]:
perguntas = [
    # "Qual o tema do documento?",
    "Quais informações importantes aparecem nele?"
    "Qual técnica possibilita o uso de ambos os protocolos?"
]

respostas = retriever.batch(perguntas)

for r in respostas:
    print(r)

# -----------------------------------------------

# Melhorando as Respostas às Perguntas com Hybrid Search

In [ ]:
!pip install langchain langchain-community langchain-huggingface sentence-transformers rank_bm25 pypdf langchain-openai

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_huggingface-1.2.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached rank_bm25-0.2.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached pypdf-6.9.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_openai-1.1.12-py3-none-any.whl.metadata (3.1 kB)
  Using cached langchain_classic-1.0.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached requests-2.33.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached langchain_core-1.2.22-py3-none-any.whl.metadata (4.4 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Us

In [ ]:
# Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI


from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

import numpy as np
import re

In [ ]:
#Carregar doc
file_path = "/content/TCC  versão 4.6.pdf"

loader = PyPDFLoader(file_path)
docs = loader.load()

print("Páginas carregadas:", len(docs))

Páginas carregadas: 73


In [ ]:
# Dividir em Chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

all_splits = text_splitter.split_documents(docs)

print("Total de chunks:", len(all_splits))

Total de chunks: 374


In [ ]:
# Embedding (SERAFIM)
embeddings = HuggingFaceEmbeddings(
    model_name="PORTULAN/serafim-900m-portuguese-pt-sentence-encoder"
)

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

In [ ]:
# Vector Store
vector_store = InMemoryVectorStore(embeddings)

vector_store.add_documents(all_splits)

print("Vector store pronto")

Vector store pronto


In [ ]:
# TOKENIZAÇÃO
def tokenize(text):
    return re.findall(r'\w+', text.lower())

In [ ]:
# BM25 (Busca Lexical)
corpus = [doc.page_content for doc in all_splits]
tokenized_corpus = [tokenize(doc) for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 pronto")

BM25 pronto


In [ ]:
# QUANTIZATION + CROSS ENCODER
from sentence_transformers import CrossEncoder
import torch

cross_encoder = CrossEncoder("./reranker-finetuned-pro",
    device="cpu"
)

cross_encoder.model = torch.quantization.quantize_dynamic(
    cross_encoder.model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

print("✅ CrossEncoder fine-tuned + quantizado carregado")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_7489/666186692.py:9: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  cross_encoder.model = torch.quantization.quantize_dynamic(


✅ CrossEncoder fine-tuned + quantizado carregado


In [ ]:
# Dataset de Avaliação
eval_data = [
    {"query": "Qual é o tema do documento?", "relevant": ["IPv4", "IPv6", "interoperabilidade"]},
    {"query": "O que é IPv6?", "relevant": ["IPv6"]},
    {"query": "Qual o objetivo do trabalho?", "relevant": ["objetivo"]}
]

In [ ]:
# MÉTRICAS (precision, recall, mrr)
def precision_at_k(results, relevant_terms, k=3):
    top_k = results[:k]
    hits = 0

    for doc in top_k:
        text = doc.page_content.lower()
        if any(term.lower() in text for term in relevant_terms):
            hits += 1

    return hits / k


def recall_at_k(results, relevant_terms):
    found_terms = set()

    for doc in results:
        text = doc.page_content.lower()
        for term in relevant_terms:
            if term.lower() in text:
                found_terms.add(term.lower())

    return len(found_terms) / len(relevant_terms)


def mrr(results, relevant_terms):
    for i, doc in enumerate(results):
        text = doc.page_content.lower()
        if any(term.lower() in text for term in relevant_terms):
            return 1 / (i + 1)
    return 0

In [ ]:
# Hybrid Search
def hybrid_search(query, k=15):
    dense_docs = vector_store.similarity_search(query, k=k)

    tokenized_query = tokenize(query)
    bm25_scores = bm25.get_scores(tokenized_query)

    top_bm25_idx = sorted(
        range(len(bm25_scores)),
        key=lambda i: bm25_scores[i],
        reverse=True
    )[:k]

    bm25_docs = [all_splits[i] for i in top_bm25_idx]

    combined = list({id(doc): doc for doc in dense_docs + bm25_docs}.values())

    return combined

In [ ]:
# Fine Tuning
# ============================================
# 🔥 FINE-TUNING PROFISSIONAL (AUTO DATASET)
# ============================================

from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader
import random

# 🔥 Queries (suas perguntas)
queries = [
    "Qual a técnica que permite a coexistência de IPv4 e IPv6?",
    "Como é denominado o transporte de IPv6 em IPv4?",
    "Como IPv6 comunica com IPv4?",
    "O que é NAT?",
    "Como funciona o tunelamento?",
    "Quais tipos de tunelamento existem?"
]

# 🔥 Criar dataset automaticamente
train_examples = []

for query in queries:
    # 🔥 Buscar candidatos reais do seu sistema
    docs = hybrid_search(query, k=20)

    if len(docs) < 5:
        continue

    # 🔥 POSITIVO (top 1)
    positive_doc = docs[0].page_content

    # 🔥 HARD NEGATIVES (top 2-5)
    negatives = [doc.page_content for doc in docs[2:6]]

    # adicionar positivo
    train_examples.append(
        InputExample(texts=[query, positive_doc], label=1.0)
    )

    # adicionar negativos difíceis
    for neg in negatives:
        train_examples.append(
            InputExample(texts=[query, neg], label=0.0)
        )

# 🔥 Dataloader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)

print(f"Total de exemplos de treino: {len(train_examples)}")

# 🔥 Modelo base
model = CrossEncoder("BAAI/bge-reranker-base", num_labels=1)

# 🔥 Treinamento
model.fit(
    train_dataloader=train_dataloader,
    epochs=2,
    warmup_steps=10,
    show_progress_bar=True
)

# 🔥 Salvar modelo
model.save("reranker-finetuned-pro")

print("✅ Fine-tuning profissional concluído!")

Total de exemplos de treino: 30


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Fine-tuning profissional concluído!


In [ ]:
# LLM AS A JUDGE
import os

# Configure sua chave de API do OpenAI aqui
# Substitua "SUA_CHAVE_AQUI" pela sua chave real
os.environ["OPENAI_API_KEY"] = ""

judge_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

def avaliar_doc(query, doc):
    prompt = f"""
Você é um avaliador de relevância em um sistema de busca.

Pergunta: {query}

Trecho:
{doc.page_content}

Responda SOMENTE com um número de 0 a 10:

0 = irrelevante
5 = parcialmente relevante
10 = totalmente relevante
"""

    resposta = judge_llm.invoke(prompt).content.strip()

    try:
        return float(resposta)
    except:
        return 5.0


def aplicar_judge(query, docs):
    avaliados = []

    for doc in docs:
        score = avaliar_doc(query, doc)
        avaliados.append((score, doc))

    return sorted(avaliados, key=lambda x: x[0], reverse=True)

In [ ]:
# RerankER
def rerank(query, docs):
    pairs = [(query, doc.page_content) for doc in docs]
    scores = cross_encoder.predict(pairs)

    boosted = []
    for score, doc in zip(scores, docs):
        boost = 0

        if "ipv6" in doc.page_content.lower():
            boost += 0.1
        if "interoperabilidade" in doc.page_content.lower():
            boost += 0.1

        boosted.append((score + boost, doc))

    return sorted(boosted, key=lambda x: x[0], reverse=True)

In [ ]:
# Filtro
def filtrar_top(reranked, min_score=0.2, top_k=5):
    filtrados = [doc for score, doc in reranked if score > min_score]
    return filtrados[:top_k]

In [ ]:
# Query Expansion
def expand_query(query):
    return [
        query,
        "tema principal do documento",
        "assunto do trabalho",
        "sobre o que é o texto",
        "qual o título do documento",
        "objetivo do documento"
    ]

In [ ]:
# Pipeline Final
def buscar_resposta(query):
    queries = expand_query(query)

    candidatos = []
    candidatos.extend(all_splits[:5])  # boost início

    for q in queries:
        candidatos.extend(hybrid_search(q, k=10))

    candidatos = list({id(doc): doc for doc in candidatos}.values())

    reranked = rerank(query, candidatos)

    top_docs = filtrar_top(reranked)

    # 🔥 evitar judge em perguntas genéricas
    if "tema" in query.lower() or "assunto" in query.lower():
        return top_docs[:3]

    # julgados = aplicar_judge(query, top_docs)

    # final_docs = [doc for score, doc in julgados if score > 6]

    return top_docs[:3]

In [ ]:
# AVALIACAO
def avaliar_pipeline():
    precisions = []
    recalls = []
    mrrs = []

    for item in eval_data:
        query = item["query"]
        relevant = item["relevant"]

        results = buscar_resposta(query)

        precisions.append(precision_at_k(results, relevant))
        recalls.append(recall_at_k(results, relevant))
        mrrs.append(mrr(results, relevant))

    print("Precision@3:", sum(precisions) / len(precisions))
    print("Recall:", sum(recalls) / len(recalls))
    print("MRR:", sum(mrrs) / len(mrrs))

In [ ]:
# INTERFACE
def detectar_titulo(doc):
    texto = doc.page_content.lower()

    # 🔥 1. tentar padrão explícito (melhor caso)
    if "título da monografia" in texto:
        linhas = doc.page_content.split("\n")
        for l in linhas:
            if "TÍTULO DA MONOGRAFIA" in l:
                return l.split(":")[-1].strip()

    # 🔥 2. fallback: procurar linhas com ipv4/ipv6
    linhas = doc.page_content.split("\n")
    for l in linhas:
        if "ipv4" in l.lower() and "ipv6" in l.lower():
            return l.strip()

    # 🔥 3. fallback final
    return linhas[0] if linhas else ""
def responder(query):
    docs = buscar_resposta(query)

    print(f"\nPergunta: {query}\n")

    if not docs:
        print("Nenhuma informação relevante encontrada.")
        return

    # 🔥 usar a função correta aqui
    titulo = extrair_titulo(docs)

    print("Possível tema/título:")
    print(titulo)
    print("\n")

    for i, doc in enumerate(docs):
        print(f"--- Trecho {i+1} ---\n")
        print(doc.page_content[:500])
        print("\n")

def extrair_titulo(docs):
    for doc in docs:
        texto = doc.page_content

        # 🔥 caso ideal
        if "TÍTULO DA MONOGRAFIA" in texto:
            for linha in texto.split("\n"):
                if "TÍTULO DA MONOGRAFIA" in linha:
                    return linha.split(":")[-1].strip()

    # 🔥 fallback: procurar IPv4 + IPv6
    for doc in docs:
        for linha in doc.page_content.split("\n"):
            if "ipv4" in linha.lower() and "ipv6" in linha.lower():
                return linha.strip()

    return "Tema não identificado"

In [ ]:
# EXECUÇÃO
responder("Qual é o tema do documento?")
avaliar_pipeline()



Pergunta: Qual é o tema do documento?

Possível tema/título:
INTEROPERABILIDADE IPV4-IPV6.


--- Trecho 1 ---

Capítulo 5 
 
  CONCLUSÃO 
5.1        Considerações finais 
Posto que o tema do trabalho era mostrar mecanismos de interoperabilidade IPv4-IPv6, esse foi iniciado 
com uma introdução ao protocolo IPv6, comparando alguns de seus aspectos com o protocolo IPv4. Neste 
ponto,  vislumbrou-se  as  principais  e  significativas  mudanças  ocorridas  no  IPv6  com  relação  a  seu 
antecessor, como o aumento do tamanho do endereçamento, que passa a ser agora de 128 bits e também


--- Trecho 2 ---

Capítulo 5 
 
  CONCLUSÃO 
5.1        Considerações finais 
Posto que o tema do trabalho era mostrar mecanismos de interoperabilidade IPv4-IPv6, esse foi iniciado 
com uma introdução ao protocolo IPv6, comparando alguns de seus aspectos com o protocolo IPv4. Neste 
ponto,  vislumbrou-se  as  principais  e  significativas  mudanças  ocorridas  no  IPv6  com  relação  a  seu 
antecessor, com